# MCMC baseline — independent-optimizer calibration (cardio-only, SI stack)

<details>
<summary>Answers the EFC reviewers' #1 ask: an **independent optimizer** for the same 16 calibration</summary>

parameters, to (a) check EFC hits its targets as well as a standard method and (b) reveal the
**identifiability** of the fit — is EFC's converged point one of many on a manifold?

Method: **affine-invariant ensemble MCMC** (`emcee`) over the 16 params, gradient-free
(event-triggered valves/HR break autodiff). Each likelihood eval is a **controllers-off**
forward sim at a fixed θ — `runnerBatchSI.batchedCalibration` run with a single
`multiplierC = 0.0` settle stage: it overlays the `(N,P)` θ-matrix onto `Y0` and returns
`rawObs (N, nObs)` (the observables are model-internal cycle-operator states already latched at
each beat — no post-hoc extraction). Walker positions **are** the θ-matrix, so one call = one
vmapped ensemble step.

**Why controllers-off calibration and not `mode="baseline"`:** `multiplierC=0` freezes only the
16 *cubic tuning knobs*; the `t_Sys_HC` **`polynomialController`** (systolic-timing-vs-heart-rate,
a *structural* physiological law, not a tuning knob) stays active. Genuine `mode="baseline"` drops
the whole calibration section, so `t_Sys_HC` freezes at the wrong default (0.4 vs ~0.56) and the
forward eval lands ~18% off the targets. The controllers-off calibration reproduces θ_EFC to
~2.1% — the intrinsic cardiac-cycle phase-jitter floor — and is consistent with the convergence /
batch runs, which also drive `batchedCalibration`.

**Structure (like `Convergence_Run.ipynb`): Phase 1 runs the sampler and writes an `.h5`;
Phase 2 loads that file and plots.** After a kernel restart, run the cheap setup cells + the
Phase-2 load cell to re-plot without re-sampling.

**Prior = the controller *numerical-safeguard* bounds** (`config/models/*.json`
`calibration[p].params.{minValue,maxValue}`), NOT the `convergence` LHS ranges. The latter are
initial-guess sampling ranges; EFC drives some params outside them (e.g. `E_Hl → ~19` vs LHS
`[1,15]`), so a prior on the LHS ranges would truncate the target-consistent region. The
safeguards are the hard bounds EFC actually operates within.

**Self-contained — no EFC dependency.** The notebook runs, saves, and plots without reading any
EFC result: a self-contained warmup (score an LHS batch with the objective, keep the best-fitting
points) supplies BOTH the whitening preconditioner and — with `init="warmup"` — the walker seeds.
The cross-technique comparison against EFC lives in `run_test/calibration_compare.ipynb`.

**Convergence, not plateau.** The archived 256-walker run stopped at step 1900 on an
error-plateau test while split-R̂ was still 2.58: the MAP was good (2.1%) but the posterior *mean*
was meaningless (201%) because un-mixed walkers were still migrating in from the prior box. The
sampler now seeds from the warmup cloud, proposes with a DE/snooker mixture, burns 10τ, and stops
only when max split-R̂ < `rhatTol` **and** min N_eff > `neffMin`.

Every knob lives in the `runConfig` dict below (project single-config-surface rule).

</details>

In [ ]:
# region -> runConfig (the ONE place run configuration lives; project single-config-surface rule)
runConfig = {
    # --- file references ---------------------------------------------------
    "model":    "cvModel_cubic.json",   # cardio-only model (16 calibration knobs + t_Sys_HC)
    "scenario": "sepsis_cubic.json",    # shared.twin + convergence.{parameters,observations}
    "mode":     "calibration",    # forward eval = a controllers-OFF calibration solve

    # --- device / precision (applied in the Imports cell, before `import jax`) ---------
    "device": {
        "useGpu":    False,       # CPU run (GPU gives no benefit here)
        "precision": "float64",   # "float64" reference | "float32" faster on GPU
    },

    # --- inference / fit backend (method-selectable over ONE forward eval + objective) ---
    "inference": {
        "method":      "emcee",   # fixed: emcee-only notebook (drives the output suffix)
        "run":         False,      # False -> plot-only: skip Phase 1 (sampler); reload the saved .h5 and plot
        # whitening: self-contained warmup preconditioner (no EFC dependency).
        "whiten": {
            "enabled": True,      # on/off (map unit coords by the warmup ridge covariance)
            "warmup":  512,       # # LHS warmup draws scored by log_prob to estimate the ridge covariance
            "keep":    0.2,       # top fraction by logp whose covariance defines the whitening map
        },
        "seed":        0,          # init + backend reproducibility
        # walker init: "warmup" seeds from the kept warmup cloud (still dispersed -- it spans the
        # ridge -- but already ON the target-consistent region, so walkers don't spend the run
        # migrating in from the box corners); "lhs" = the old full-box LHS seeding.
        "init":        "warmup",   # "warmup" | "lhs"
        "initJitter":  0.05,       # resample / rescue jitter, as a fraction of the source cloud's sd
        "settleRuns":  3,          # forward runs to settle the cycle before reading obs
        "sigmaRel":    0.005,       # relative likelihood scale: sigma_i = sigmaRel * |target_i|
        "chunkSize":   128,         # samples per vmapped solve (bounds VRAM)
        "printEvery":  25,         # print diagnostic every N iters (0 = tqdm bar only)

        # --- emcee (affine-invariant ensemble MCMC) ------------------------
        "emcee": {
            "nWalkers":  128,       # ensemble size (must be > 2*P); steps buy mixing, walkers past ~4-8*P do not
            "nSteps":    1500,     # sampler iterations (a CAP when earlyStop is on) -- ~40 tau at tau ~ 200
            "burnIn":    750,     # steps discarded before flattening (~10 tau; 400 was ~2 tau and left the transient in)
            "move":      "de",     # "de" -> DEMove/DESnookerMove mixture (recommended >~10 dims) | "stretch" -> StretchMove
            "snookerFrac": 0.2,    # fraction of the "de" mixture given to the anti-trapping snooker move
            "stretchA":  1.7,      # StretchMove scale a -- ONLY used when move == "stretch" (archived run used 2.0)
            # burn-in walker rescue: DE proposal sizes are set by the spread of the OTHER walkers, so
            # a walker that lands far off can never random-walk back. The archived 8000-step run had
            # exactly one (walker 36: J ~ 22681, only 4 moves in 6000 steps) -- it alone held max
            # split-Rhat at 441.7 (1.006 without it), blocked the early stop (~5 h of wasted compute)
            # and dragged the reported ensemble mean|rel| from 1.39% to 2.42%. Any walker whose logp
            # sits `logpDrop` below the ensemble median is reseeded from a random healthy walker plus
            # `inference.initJitter` x the ensemble sd. BURN-IN ONLY -- the sampling phase stays
            # untouched, detailed-balance MCMC.
            "rescue": {
                "enabled":  True,
                "logpDrop": 50.0,  # logp this far below the median = stranded (posterior logp fluctuates ~P/2 = 8)
                "every":    50,    # steps between rescue sweeps
            },
            # early stop: quit ONLY on a convergence criterion -- max split-Rhat < rhatTol AND
            # min N_eff > neffMin over the post-burn chain, both read in physical coords exactly as
            # the Phase-2 diagnostics cell reports them. The old plateau test (fit error + param
            # centre stopped moving) is satisfied by a STUCK, unmixed ensemble and truncated the
            # archived run at step 1900 with split-Rhat 2.58.
            "earlyStop":  True,    # stop once converged; nSteps is then only the cap
            "checkEvery": 250,     # steps between convergence checks (each costs an autocorr estimate)
            "rhatTol":    1.05,    # max split-Rhat below this = chains agree
            "neffMin":    400,     # min N_eff above this = enough independent samples
            "minSteps":   1500,    # never stop before this (must exceed burnIn)
        },
    },

    # --- output (name is the STEM; method appended -> mcmc_baseline_sepsis_<method>.h5) ----
    "output": {
        "save": True,
        "path": "notebookData/convergence",
        "name": "mcmc_baseline2_sepsis.h5",
    },

    # --- paper artifacts (appendix table + figure; flip emit True for one run, then revert) ---
    "paper": {
        "emit":        False,     # True -> write the .tex table and the .png figure, then revert
        "dir":         "EFC_Paper/revision/generated",   # generated LaTeX fragments
        "imageDir":    "EFC_Paper/revision/Images",      # generated figures
        "table":       "mcmcSummary.tex",                # non-float table -> Appendix (app:mcmc)
        "figure":      "mcmcTraces.png",                 # 6-panel search-trace figure
        # The four parameters that separate the gradient-descent baseline's two basins (see
        # gd.ipynb). The SAME four are drawn for both methods so the two appendix figures read
        # side by side -- here they show the sampler never visits the spurious basin at all.
        "traceParams": ["R_As_Cs", "V0_As", "C_As", "E_Hl"],
        "scatterPair": ["R_As_Cs", "V0_As"],  # final-point scatter plane
        "errBand":     2.0,          # % reference line on the residual panel
        # 1:1 at the manuscript's 8.5 cm column (3x2 grid): a wider figure downscaled into
        # the same column renders its 6-8 pt annotations unreadably small.
        "figSize":     [3.4, 5.2],   # inches, included at \includegraphics[width=8.5cm]
        "dpi":         200,
    },

    # --- analysis / plot ---------------------------------------------------
    "analysis": {
        "atm":             760.0,   # atmospheric offset for absolute-pressure signals
        "divergenceLimit": 1e6,     # |value| >= this in any observable/param = out of scope
        "errBand":         0.5,     # +/- target-tolerance band drawn on the per-observation boxplot (%)
        "stepTraceYLim":   "initial",  # step-trace panel y-limits: "initial" (frame to iter-0 spread) | "robust" | None
        "stepTracePct":    [1, 99],    # robust percentile for the step-trace y-limit clip
        "nTraceParams":    6,       # walker-trace panels: N loosest-mixing params (mixing diagnostics)
    },

    # --- required by runner.buildSimulationParams (unused here) ------------
    "plots": [],
    "printStatus": False,

    # --- integration numerics (override scenario shared.integration) -------
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 1.0,      # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

<details>
<summary>Device/precision must be set from `runConfig` **before** `import jax`.</summary>



</details>

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
from scipy.stats import qmc                      # Latin Hypercube sampling
import pandas as pd
import matplotlib.pyplot as plt
import emcee                                     # affine-invariant ensemble sampler (method="emcee")
import corner                                    # posterior marginals + pairwise
import h5py
import json
import time

import library.run.runner as runner              # buildSimulationParams
import library.run.stateSetup as stateSetup   # resolveCalibrationBounds (single-source calibration ranges)
import library.run.runnerBatchSI as runnerBatchSI  # batched (vmapped) forward solve
import library.run.progress as progressLib       # standard live-progress line + record log
import library.viz.plots as libPlots             # plotCalibrationConvergence (step traces)
import library.utils as utils
from library.hdf5 import schema_pop               # write the per-run progress log into the saved .h5

print(f"jax {jax.__version__} | emcee {emcee.__version__} | "
      f"x64={jax.config.jax_enable_x64} | devices={jax.devices()}")
# endregion

## Setup — `simulationParams`, prior bounds, targets

<details>
<summary>Cheap, scenario-only cells (no sampling). Re-run these + the Phase-2 load cell after a kernel</summary>

restart to re-plot a saved run. `buildForwardParams` swaps the scenario's staged calibration for
a single `multiplierC = 0.0` settle stage: the controller gain is clamped to zero so the 16
param-states stay frozen at their `Y0` values — a pure forward evaluation at θ.

</details>

In [ ]:
# region -> setup: simulationParams, prior bounds, forward evaluator
scenario = utils.loadScenario(runConfig["scenario"])
model    = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))
conv         = scenario["convergence"]
calConf      = runner.buildSimulationParams(runConfig, scenario)["simulationConf"]["calibration"]
twin         = scenario["shared"]["twin"]["twinTargets"]
volDist      = scenario["shared"]["twin"]["volumeDistribution"]
param_names  = calConf["adaptive"]["parameters"]
observations = conv["observations"]
P            = len(param_names)
ic           = runConfig["inference"]
method       = ic["method"]
# output name carries the method so the three backends' runs coexist on disk
_stem, _ext  = os.path.splitext(runConfig["output"]["name"])
outPath      = os.path.join(runConfig["output"]["path"], f"{_stem}_{method}{_ext or '.h5'}")

# Prior bounds = the resolved calibration range (model min/max, overridable per scenario/runConfig):
# the hard limit EFC operates within and the single source of truth for the box — the correct prior.
# Same resolveCalibrationBounds the solve-time clamp and the convergence seeding box use.
bounds = stateSetup.resolveCalibrationBounds(model["calibration"], calConf, param_names)
lo, hi = bounds[:, 0], bounds[:, 1]
span   = hi - lo


def buildForwardParams(settleRuns):
    """simulationParams whose calibration is one controllers-off (multiplierC=0) settle stage.
    buildSimulationParams returns `simulationConf` as the SAME scenario dict by reference, so we
    shallow-copy it and swap only `calibration` — else this clobbers the shared scenario."""
    sp = runner.buildSimulationParams(runConfig, scenario)
    conf = dict(sp["simulationConf"])
    conf["calibration"] = {
        "strategy": "staged",
        "maxCubicFactor": scenario["calibration"].get("maxCubicFactor", 10.0),
        "stages": [{
            "description": "MCMC baseline settle (controllers off)",
            "runsToIgnore": int(settleRuns), "runsToSave": 0,
            "multiplier": 0.0, "multiplierC": 0.0,
            "parameters": param_names, "targets": {}, "states": {},
        }],
    }
    sp["simulationConf"] = conf
    # silence the batched-solve progress line inside each forward eval (thousands of evals); the
    # live view for MCMC is the per-step sampler line in the backend loop (runEmcee etc.).
    sp["solver"] = {**sp["solver"], "progressEvery": 0}
    return sp


def toUnit(theta):      # physical -> unit [0,1]^P
    return (np.atleast_2d(np.asarray(theta, dtype=float)) - lo) / span

def toPhysical(u):      # unit -> physical
    return lo + np.atleast_2d(np.asarray(u, dtype=float)) * span


sp   = buildForwardParams(ic["settleRuns"])
prep = runnerBatchSI.prepare(sp)                 # scalar setup once; reused every backend iteration

def forwardWith(sp_, prep_, theta):
    """theta (P,) or (n,P) -> full batchedCalibration result (rawObs (n,nObs) + finalStates)."""
    return runnerBatchSI.batchedCalibration(sp_, np.atleast_2d(np.asarray(theta, dtype=float)),
            param_names, observations, chunkSize=ic["chunkSize"], prepared=prep_, printStatus=False)

print(f"P = {P} params | observations = {len(observations)} | method = {method} | "
      f"prior = calibration bounds | settleRuns = {ic['settleRuns']} (runTime = {sp['runTime']} s each)")
print(f"output -> {outPath}")
# endregion

In [ ]:
# region -> twin targets, atmospheric offsets, likelihood sigma
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted. The targeted set is
    EFC's calibration set: the observables its cubic controllers drive (model['calibration']
    varTargets). Capillary means are a pressure DROP from the upstream arterial target, not
    absolute means. NOT targets (no controller drives them, so EFC never fits them):
      - Cyc_HC : cycle period = 60/HR, set by the driver (an INPUT).
      - V_Vs   : systemic venous volume -- the slack compartment absorbing the remaining blood
                 volume; no calibration controller targets it."""
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        "keep_SV_Hl": twin["CO"] / twin["HR"],
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
targeted  = ~np.isnan(targetArr)                 # observations that have a twin target
tgt       = targetArr[targeted]                  # (nTargeted,) gauge-unit targets
sigma     = ic["sigmaRel"] * np.abs(tgt)         # single relative scale: sigma_i = sigmaRel*|t_i|
obsTargeted = [o for o, t in zip(observations, targeted) if t]
pd.DataFrame({"observation": obsTargeted, "target": tgt, "sigma": sigma})
# endregion

# Phase 1 — Run & save

<details>
<summary>Everything below samples the posterior and writes `outPath`. Skip to Phase 2 to re-plot a</summary>

previously saved file.

</details>

## Objective / likelihood / prior

<details>
<summary>Explicit objective (the definition reviewer #2 asked for):</summary>

$$J(\theta) = \sum_i \left(\frac{o_i(\theta) - t_i}{\sigma_i}\right)^2, \qquad \sigma_i = \varepsilon\,|t_i|.$$
Gaussian likelihood $\propto \exp(-\tfrac12 J)$; **uniform prior** over the safeguard bounds.
The sampler works in **unit space** $u\in[0,1]^P$ (θ = lo + u·span): the 16 params span ~0.02–450,
and the affine-invariant stretch move is scale-sensitive, so raw physical coordinates are
ill-conditioned (emcee rejects them). `emcee` with `vectorize=True` hands `log_prob` an `(n, P)`
block — one `batchedCalibration` call scores them all.

</details>

In [ ]:
# region -> log-posterior + prior (unit space) with in/out-of-bounds sanity checks
def log_prob(u):
    """Vectorized log-posterior in UNIT space u in [0,1]^P. u: (n,P) ->
    list of per-walker (logp, obsBlob): logp is the scalar log-posterior; obsBlob is the
    (nTargeted,) gauge-unit observation vector, captured by emcee as a blob so every step's
    observations are available for the step-trace plots (no recompute in Phase 2).
    Uniform prior (inside the box) x Gaussian likelihood exp(-J/2); diverged lanes (non-finite
    or astronomically large obs) get logp = -inf cleanly."""
    u = np.atleast_2d(np.asarray(u, dtype=float))
    # tol absorbs the whitening round-trip's ~1e-16 nudge at the [0,1] safeguard bound (emcee/SMC
    # sample in whitened coords, where whiteToUnit can push a boundary particle a hair out-of-box);
    # clip then scores it AT the bound instead of dropping it to -inf. Genuine out-of-box draws
    # (past tol) still fail the check -> logp = -inf below.
    inb = np.all((u >= -1e-9) & (u <= 1.0 + 1e-9), axis=1)
    u   = np.clip(u, 0.0, 1.0)
    obs = (forwardWith(sp, prep, lo + u * span)["rawObs"] - offsetArr)[:, targeted]  # (n,nTargeted) gauge
    finite = np.all(np.isfinite(obs), axis=1) & np.all(np.abs(obs) < 1e12, axis=1)
    with np.errstate(over="ignore", invalid="ignore"):
        ll = -0.5 * np.sum(((obs - tgt) / sigma) ** 2, axis=1)
    good = inb & finite & np.isfinite(ll)
    ll = np.where(good, ll, -np.inf)
    return list(zip(ll, obs))                       # per-walker (logp, obs-blob)

def logp_only(theta):                               # scalar log-posterior for checks / physical theta
    return np.array([r[0] for r in log_prob(toUnit(np.atleast_2d(theta)))])

if runConfig["inference"].get("run", True):
    # sanity: in-bounds draws finite, out-of-bounds -inf
    _chk = qmc.scale(qmc.LatinHypercube(d=P, seed=123).random(4), lo, hi)
    print("log_prob(in-bounds draws):", np.round(logp_only(_chk), 2))
    print("log_prob(out-of-bounds)  :", np.array([r[0] for r in log_prob(np.full((1, P), 1.5))]))
# endregion

In [ ]:
# region -> preconditioning: warmup-covariance whitening + shared population init helper
if runConfig["inference"].get("run", True):
    # ONE self-contained warmup cloud serves two jobs (no EFC dependency): draw `warmup` LHS unit
    # points, score them with the existing objective (log_prob, cell above), keep the top `keep`
    # fraction by logp -- that cloud IS the ridge estimate.
    #   (1) WHITENING: reparametrise the unit posterior by the cloud's covariance so the curved
    #       target ridge becomes ~isotropic in whitened coords w (u = uMean + w @ L.T,
    #       L = chol(cov)). The ensemble moves assume roughly isotropic geometry, so this removes
    #       the migration / mixing cost along the ridge. The map is AFFINE -> with a uniform box
    #       prior the target is unchanged up to a constant Jacobian (no correction term).
    #   (2) INIT (init="warmup"): seed the walkers FROM that cloud instead of from full-box LHS.
    #       Full-box LHS in 16-D starts most walkers many ridge-sd out; they then spend the run
    #       migrating in, and that transient -- not a competing mode -- is what wrecked the
    #       archived run's posterior mean (MAP 2.1% vs mean 201%).
    # CMA-ES adapts its own covariance in raw unit coords, so it disables whitening (whiten.enabled=False).
    wc = ic.get("whiten", {})
    initMode = ic.get("init", "warmup")
    warmCloud = None
    if wc.get("enabled", True) or initMode == "warmup":
        nWarm = wc.get("warmup", 512); keep = wc.get("keep", 0.2)
        uWarm  = qmc.LatinHypercube(d=P, seed=ic["seed"]).random(nWarm)    # (nWarm, P) unit-space warmup
        lpWarm = np.array([r[0] for r in log_prob(uWarm)])                 # score each draw with the objective
        nSel   = max(P + 1, int(np.ceil(keep * nWarm)))                    # floor at P+1 for a full-rank cov
        warmCloud = uWarm[np.argsort(-lpWarm)[:nSel]]                      # top `keep` fraction by logp
        print(f"warmup: {nWarm} LHS draws scored -> kept best {nSel} | "
              f"logp best {np.nanmax(lpWarm):.4g}, kept-worst {np.sort(lpWarm)[-nSel]:.4g}")

    if wc.get("enabled", True):
        sel   = warmCloud
        uMean = sel.mean(0)
        cov   = np.cov(sel.T) + 1e-9 * np.eye(P)          # ridge-jitter -> positive-definite Cholesky
        L     = np.linalg.cholesky(cov)
        Linv  = np.linalg.inv(L)
        def whiteToUnit(w): return uMean + np.atleast_2d(np.asarray(w, float)) @ L.T
        def unitToWhite(u): return (np.atleast_2d(np.asarray(u, float)) - uMean) @ Linv.T
        print(f"whitening ON | cov cond {np.linalg.cond(cov):.1f} | "
              f"stdev [{np.sqrt(np.diag(cov)).min():.3g}, {np.sqrt(np.diag(cov)).max():.3g}]")
    else:
        def whiteToUnit(w): return np.atleast_2d(np.asarray(w, float))
        def unitToWhite(u): return np.atleast_2d(np.asarray(u, float))
        print("whitening OFF | sampling raw unit coords")

    def logpW(w):
        """log_prob in whitened coords: w -> u -> the unit-space (logp, obsBlob) list (emcee / SMC reuse)."""
        return log_prob(whiteToUnit(w))

    def initUnit(n, rng):
        """n unit-space [0,1]^P init points, per `inference.init`.
        "warmup": a random subset of the kept warmup cloud -- dispersed (the cloud spans the ridge)
        but already on the target-consistent region. If n exceeds the cloud size it resamples with
        replacement and adds `initJitter` x the cloud sd so walkers stay distinct (emcee requires it).
        "lhs": LHS over the full prior box -- the fully-naive seeding the archived run used."""
        if initMode == "warmup" and warmCloud is not None:
            m = warmCloud.shape[0]
            if n <= m:
                return warmCloud[rng.choice(m, size=n, replace=False)]
            jitter = ic.get("initJitter", 0.05) * warmCloud.std(axis=0)
            pts = warmCloud[rng.choice(m, size=n, replace=True)] + rng.normal(scale=jitter, size=(n, P))
            return np.clip(pts, 0.0, 1.0)
        return qmc.LatinHypercube(d=P, seed=ic["seed"]).random(n)          # lhs: disperse over the box
else:
    print("plot-only: preconditioning/whitening skipped (only needed to seed the sampler)")
# endregion

In [ ]:
# region -> backend: emcee (affine-invariant ensemble MCMC over the whitened posterior)
def buildMoves(ec):
    """emcee move set from runConfig. "de" (default) = the DEMove / DESnookerMove mixture
    recommended above ~10 dims: differential-evolution proposals follow the ensemble's own
    correlation structure, and the snooker component is the anti-trapping move. "stretch" = the
    classic affine-invariant StretchMove (what the archived 256-walker run used -- select it to
    stay directly comparable to that file)."""
    if ec.get("move", "de") == "stretch":
        return emcee.moves.StretchMove(a=ec.get("stretchA", 2.0))
    f = float(ec.get("snookerFrac", 0.2))
    return [(emcee.moves.DEMove(), 1.0 - f), (emcee.moves.DESnookerMove(), f)]


def mixing(postW):
    """(max split-Rhat, min N_eff) over the post-burn chain. postW is (nPost, nW, P) in WHITENED
    coords; it is mapped back to PHYSICAL first so these numbers mean exactly what the Phase-2
    diagnostics cell reports. NaN (-> never triggers the stop) while there is too little history."""
    if postW.shape[0] < 2 * P:
        return float("nan"), float("nan")
    post = toPhysical(whiteToUnit(postW.reshape(-1, P))).reshape(postW.shape)
    rhat = np.array([utils.splitRhat(post[:, :, j]) for j in range(P)])
    try:
        tau  = np.asarray(emcee.autocorr.integrated_time(post, tol=0), float)
        nEff = float(np.nanmin(post.shape[0] * post.shape[1] / tau))
    except Exception:
        nEff = float("nan")
    return float(np.nanmax(rhat)), nEff


def rescueStranded(state, rng, logpDrop):
    """Reseed walkers stranded far from the ensemble, IN PLACE on the sampler's live state.

    emcee's DE proposals are displacements built from the difference of two OTHER walkers, so their
    size is set by the ensemble's spread: a walker sitting orders of magnitude below the rest in
    log-posterior proposes steps far too small to walk back, and stays parked for the whole run
    (archived file: walker 36, 4 accepted moves in 6000 steps). Such a walker single-handedly holds
    split-Rhat far from 1 and skews every ensemble-mean readout.

    A walker is stranded if its logp is non-finite or sits `logpDrop` below the ensemble median; it
    is reseeded at a random healthy walker's position plus `inference.initJitter` x the ensemble sd.
    coords / log_prob / blobs are written together so the next proposal sees a consistent state.
    Call during BURN-IN ONLY -- this is not a reversible MCMC move."""
    lpNow    = np.asarray(state.log_prob, dtype=float)
    stranded = np.where(~np.isfinite(lpNow) | (lpNow < np.median(lpNow) - logpDrop))[0]
    healthy  = np.setdiff1d(np.arange(lpNow.size), stranded)
    if stranded.size == 0 or healthy.size < 2:
        return 0
    jitter = ic.get("initJitter", 0.05) * state.coords[healthy].std(axis=0)
    newW   = (state.coords[rng.choice(healthy, size=stranded.size)]
              + rng.normal(scale=jitter, size=(stranded.size, state.coords.shape[1])))
    out = logpW(newW)                                     # rescore the reseeded walkers (a few evals)
    state.coords[stranded]   = newW
    state.log_prob[stranded] = [o[0] for o in out]
    state.blobs[stranded]    = np.asarray([o[1] for o in out], dtype=float)
    return int(stranded.size)


def runEmcee():
    """Affine-invariant ensemble MCMC (emcee) over the WHITENED posterior, walkers seeded per
    `inference.init` ("warmup" = drawn from the kept warmup cloud, i.e. already on the ridge) and
    proposed per `emcee.move` ("de" = DE/snooker mixture). Stops early ONLY on a convergence
    criterion -- max split-Rhat < rhatTol AND min N_eff > neffMin on the post-burn chain -- because
    the old plateau test (fit error + param centre stopped moving) is satisfied by a stuck, unmixed
    ensemble and truncated the archived run at step 1900 with split-Rhat 2.58. Returns the common
    tuple (chain, lp, chainSteps, obsSteps, diag, fitWall, nEval), chain / steps in PHYSICAL coords."""
    ec = ic["emcee"]; nWalkers, nSteps, burnIn = ec["nWalkers"], ec["nSteps"], ec["burnIn"]
    assert nWalkers > 2 * P, f"nWalkers must exceed 2*P = {2*P}"
    earlyStop  = ec.get("earlyStop", True)
    checkEvery = ec.get("checkEvery", 250)
    rhatTol    = ec.get("rhatTol", 1.05); neffMin = ec.get("neffMin", 400)
    minSteps   = ec.get("minSteps", 3000)
    rs = ec.get("rescue", {}); rescueOn = rs.get("enabled", True)
    logpDrop = rs.get("logpDrop", 50.0); rescueEvery = max(1, int(rs.get("every", 50)))
    nRescued = 0
    assert not earlyStop or minSteps > burnIn, "minSteps must exceed burnIn (the gate reads the post-burn chain)"
    rng = np.random.default_rng(ic["seed"])
    w0  = unitToWhite(initUnit(nWalkers, rng))            # init points, mapped into whitened coords
    print(f"emcee | init={ic.get('init', 'warmup')} move={ec.get('move', 'de')} "
          f"whiten={wc.get('enabled', True)} | walkers {w0.shape} | rescue={rescueOn} "
          f"(logpDrop {logpDrop} every {rescueEvery}, burn-in only) | earlyStop={earlyStop} "
          f"(split-Rhat<{rhatTol} & N_eff>{neffMin}; cap {nSteps})")

    sampler = emcee.EnsembleSampler(nWalkers, P, logpW, vectorize=True, moves=buildMoves(ec))
    printEvery = ic.get("printEvery", 0); t0 = time.time()
    reporter = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
    stopped = False; rhatMax = nEff = float("nan")
    for step, state in enumerate(sampler.sample(w0, iterations=nSteps, progress=True), start=1):
        if printEvery and (step % printEvery == 0 or step == nSteps):
            reporter.emit(kind="step", label=f"step {step}/{nSteps}", done=step, total=nSteps,
                          elapsedWall=time.time() - t0,
                          stats=progressLib.relErrorStats(np.asarray(state.blobs, dtype=float), tgt),
                          acc=float(np.mean(sampler.acceptance_fraction)))
        # burn-in walker rescue (see runConfig emcee.rescue): pull any walker that got stranded far
        # from the ensemble back into it, so one dead walker cannot hold split-Rhat off 1 for the
        # whole run. Gated to step < burnIn -- the sampling phase is untouched MCMC.
        if rescueOn and step < burnIn and step % rescueEvery == 0:
            nRescued += rescueStranded(state, rng, logpDrop)
        # convergence early stop: the only reason to quit before the cap is that the chains AGREE
        # (split-Rhat) and carry enough independent samples (N_eff). A stuck ensemble fails both,
        # so unlike the old plateau test this cannot mistake "not moving" for "converged".
        if earlyStop and step >= minSteps and step % checkEvery == 0:
            rhatMax, nEff = mixing(sampler.get_chain(discard=burnIn))
            if rhatMax < rhatTol and nEff > neffMin:
                stopped = True
                print(f"emcee: early stop at step {step}/{nSteps} (converged: max split-Rhat "
                      f"{rhatMax:.3f} < {rhatTol} & min N_eff {nEff:.0f} > {neffMin})")
                break
    fitWall = time.time() - t0
    nRun = sampler.iteration                              # steps actually taken (== nSteps if no early stop)
    if not stopped:                                       # report where the cap left the chain
        rhatMax, nEff = mixing(sampler.get_chain(discard=burnIn))

    chain      = toPhysical(whiteToUnit(sampler.get_chain(discard=burnIn, flat=True)))    # (S,P)
    lp         = sampler.get_log_prob(discard=burnIn, flat=True)                           # (S,)
    stepsW     = sampler.get_chain()                                                       # (nRun,nW,P) whitened
    chainSteps = toPhysical(whiteToUnit(stepsW.reshape(-1, P))).reshape(nRun, nWalkers, P)
    obsSteps   = np.asarray(sampler.get_blobs(), dtype=float)                              # (nRun,nW,nTargeted)
    acc = float(np.mean(sampler.acceptance_fraction))
    try:    tau = [float(x) for x in np.round(sampler.get_autocorr_time(tol=0), 1)]        # diagnostic only
    except Exception: tau = None
    diag = {"accept": acc, "tau": tau, "burnIn": burnIn, "nWalkers": nWalkers, "nSteps": nRun,
            "nStepsCap": nSteps, "earlyStopped": stopped, "rhatMax": rhatMax, "nEffMin": nEff,
            "move": ec.get("move", "de"), "init": ic.get("init", "warmup"),
            "nRescued": nRescued,
            "progress": reporter.records}
    print(f"emcee: {fitWall:.1f}s | acceptance {acc:.3f} | steps {nRun}/{nSteps}"
          f"{' (early stop)' if stopped else ' (cap)'} | post-burn samples {chain.shape[0]} | "
          f"max split-Rhat {rhatMax:.3f} | min N_eff {nEff:.0f} | rescued {nRescued} | tau {tau}")
    return chain, lp, chainSteps, obsSteps, diag, fitWall, nWalkers * nRun
# endregion

## Run the selected backend

<details>
<summary>`runConfig.inference.method` selects one of three gradient-free backends over the SAME</summary>

controllers-off forward evaluator + objective `J`: **emcee** (affine-invariant ensemble MCMC),
**cmaes** (CMA-ES independent optimizer + Laplace), **smc** (blackjax adaptive tempered SMC). Each
returns the same 7-tuple `(chain, lp, chainSteps, obsSteps, diag, fitWall, nEval)`, so the save and
Phase-2 plot cells never branch on method. The population is initialised per `init` (`warmup` —
drawn from the kept warmup cloud, already on the ridge; `lhs` — dispersed over the full prior box),
and emcee proposes per `emcee.move` (`de` — DE/snooker mixture; `stretch` — the classic
affine-invariant move). **Runtime scales with population × iterations — keep both small for a
smoke test; the full run is a real experiment.**

</details>

In [ ]:
# region -> run the selected backend; collect the common tuple + shared convergence-error report
if runConfig["inference"].get("run", True):
    # runEmcee scores the controllers-off forward eval with the objective J and returns the
    # common 7-tuple, so Phase-2 save/plot consumes a single, method-agnostic result shape.
    chain, lp, chainSteps, obsSteps, diag, fitWall, nEval = runEmcee()
    print(f"[{method}] chain {chain.shape} | steps {chainSteps.shape} | obs {obsSteps.shape} | "
          f"{nEval} forward evals | {fitWall:.1f}s")

    # progress trace for the persisted log: reuse the backend's live records if it kept them (emcee),
    # else reconstruct per-iteration convergence from obsSteps so every backend logs a comparable trace.
    if not diag.get("progress"):
        _pr = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
        _every = max(1, ic.get("printEvery", 25) or 1); _nIt = obsSteps.shape[0]
        for _it in range(_nIt):
            if (_it + 1) % _every == 0 or _it == _nIt - 1:
                _pr.emit(kind="step", label=f"iter {_it + 1}/{_nIt}", done=_it + 1, total=_nIt,
                         elapsedWall=float("nan"), acc=diag.get("accept"),
                         stats=progressLib.relErrorStats(obsSteps[_it], tgt), show=False)
        diag["progress"] = _pr.records

    # --- calibration error vs twin targets: watch the population collapse from its dispersed start
    # --- (iter 0) onto the targets (final iter), reported over ALL targets. obsSteps may be NaN-
    # --- padded (ragged gens) -> nan-aware reducers.
    r      = np.abs(obsSteps - tgt) / np.abs(tgt)
    meanR  = np.nanmean(r, axis=2) * 100.0                 # (iter, member) mean|rel err| %
    maxR   = np.nanmax(r, axis=2) * 100.0                  # (iter, member) max|rel err| %
    im, fm = np.nanmean(meanR[0]), np.nanmean(meanR[-1])   # ensemble-mean mean|rel|: iter 0 -> final
    ix, fx = np.nanmean(maxR[0]),  np.nanmean(maxR[-1])    # ensemble-mean  max|rel|: iter 0 -> final
    print(f"calibration error [all {len(tgt)} targets] (mean over members):")
    print(f"  mean|rel| {im:5.1f}% -> {fm:4.1f}% | max|rel| {ix:5.1f}% -> {fx:4.1f}% | "
          f"best member max|rel| {np.nanmin(maxR):.2f}%")
else:
    print("plot-only: sampler run skipped -- no MCMC; Phase 2 loads the saved posterior")
# endregion

## Save the posterior

<details>
<summary>Self-contained `.h5`: the flattened physical chain, log-prob, targets/sigma/bounds, and the</summary>

equivalence observables at MAP / mean (pre-computed so Phase 2 needs no forward solves).
`runConfig` + twin as attrs.

</details>

In [ ]:
# region -> save the fit + equivalence observables to a self-contained .h5
if runConfig["inference"].get("run", True):
    # equivalence observables (computed here so Phase 2 is pure-load). MAP/mean are fit outputs -> a
    # live forward solve of the two summary points.
    theta_MAP  = chain[np.argmax(lp)]
    theta_mean = chain.mean(axis=0)
    fitObs     = (forwardWith(sp, prep, np.array([theta_MAP, theta_mean]))["rawObs"] - offsetArr)[:, targeted]
    equivLabels = ["MAP", "mean"]
    equivObs    = fitObs                                              # (2, nTargeted); the two live-fit rows

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)
        with h5py.File(outPath, "w") as f:
            f.create_dataset("chain", data=chain, compression="gzip")
            f.create_dataset("log_prob", data=lp, compression="gzip")
            # full per-iteration trajectories (nIter, nPop, .) for the step-trace plots
            f.create_dataset("chain_steps", data=chainSteps, compression="gzip")
            f.create_dataset("obs_steps", data=obsSteps, compression="gzip")
            f.create_dataset("targets", data=tgt)
            f.create_dataset("sigma", data=sigma)
            f.create_dataset("bounds", data=bounds)
            f.create_dataset("equiv_theta", data=np.array([theta_MAP, theta_mean]))
            f.create_dataset("equiv_obs", data=equivObs)
            f.create_dataset("equiv_labels", data=np.array(equivLabels, dtype="S"))
            f.create_dataset("param_names", data=np.array(param_names, dtype="S"))
            f.create_dataset("observation_names", data=np.array(obsTargeted, dtype="S"))
            f.attrs["runConfig"] = json.dumps(runConfig)
            f.attrs["twinTargets"] = json.dumps(twin)
            f.attrs["method"] = method
            f.attrs["diag"] = json.dumps(diag)                        # per-backend diagnostics
            f.attrs["accept"] = float(diag.get("accept", float("nan")))
            f.attrs["fitWall"] = fitWall
            f.attrs["nEval"] = int(nEval)
            f.attrs["settleRuns"] = ic["settleRuns"]
            f.attrs["runTime"] = sp["runTime"]
        schema_pop.write_progress(outPath, diag.get("progress"), meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"], "method": method,
            "solver": sp["solver"]["type"], "nPop": int(obsSteps.shape[1]), "stack": "SI"})
        print(f"wrote {outPath}  (method={method}, chain {chain.shape}, steps {chainSteps.shape})")
else:
    print("plot-only: save skipped -- not overwriting the saved .h5")
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the `.h5` — no in-session sampler state required. After a</summary>

kernel restart, run the setup cells (config → imports → bounds/targets, all cheap) then this
load cell and the plot/table cells, without re-sampling.

</details>

In [ ]:
# region -> load everything the analysis needs, straight from the saved file
with h5py.File(outPath, "r") as f:
    chain       = f["chain"][:]
    lp          = f["log_prob"][:]
    chainSteps  = f["chain_steps"][:]
    obsSteps    = f["obs_steps"][:]
    tgt         = f["targets"][:]
    sigma       = f["sigma"][:]
    bounds      = f["bounds"][:]
    equivTheta  = f["equiv_theta"][:]
    equivObs    = f["equiv_obs"][:]
    equivLabels = list(f["equiv_labels"].asstr()[:])
    param_names = list(f["param_names"].asstr()[:])
    obsTargeted = list(f["observation_names"].asstr()[:])
    method      = f.attrs.get("method", "emcee")
    diag        = json.loads(f.attrs.get("diag", "{}"))
    accept      = float(f.attrs.get("accept", float("nan")))
    fitWall     = f.attrs["fitWall"]
    nEval       = f.attrs["nEval"]
    settleRuns  = f.attrs["settleRuns"]
    runTime     = f.attrs["runTime"]
lo, hi = bounds[:, 0], bounds[:, 1]
print(f"loaded {outPath}: method={method} | chain {chain.shape} | steps {chainSteps.shape} | "
      f"{nEval} forward evals | accept {accept:.3f}")
# endregion

## Step traces — observations & parameters (batch-style)

<details>
<summary>The same per-signal panels as the convergence run, but the x-axis is the **MCMC step** and each</summary>

jet line is one **walker**. Observations carry their twin **target** (dashed); parameters carry
the **prior bounds** (dotted). Burn-in is included, so you watch the ensemble start dispersed
across the prior and collapse toward the target-consistent region — the MCMC analogue of the
calibration controllers driving each observation onto its target. The spread that *remains* after
burn-in is the posterior width: tight ⇒ well-determined; a persistent band ⇒ a sloppy/
non-identifiable direction. (Reminder: this run discarded the first `burnIn` steps when forming
the corner posterior above.)

</details>

In [ ]:
# region -> step traces: one panel per targeted observation (population vs target)
# --- step traces: one panel per targeted observation (each member = a jet line vs its target) ---
# obsSteps is (nIter, nPop, nTargeted) gauge-unit observations per backend iteration (emcee step /
# cmaes generation / smc tempering stage; cmaes gens are NaN-padded to a common width). Per panel:
# every member's observation value vs iteration (jet), with its twin target (dashed). The fit
# analogue of the calibration-convergence plot: the population starts dispersed and collapses onto
# each target; the residual spread is the noise / posterior width. `model` (setup cell) maps obs ->
# controller.
obsTraces    = {o: obsSteps[:, :, k].T for k, o in enumerate(obsTargeted)}      # {obs:(nPop,nIter)}
targetsByObs = {o: float(t) for o, t in zip(obsTargeted, tgt)}
paramForObs  = {model["calibration"][p]["params"]["varTarget"]: p
                for p in param_names if p in model.get("calibration", {})}

libPlots.plotCalibrationConvergence(
    obsTraces, traceT=None, targets=targetsByObs, paramForObs=paramForObs, showLegend=False,
    ylim=runConfig["analysis"]["stepTraceYLim"], robustPct=tuple(runConfig["analysis"]["stepTracePct"]),
    divLimit=runConfig["analysis"]["divergenceLimit"],
    title=f"{method} step traces -- observations (population, all iterations)")
plt.show()
# endregion

In [ ]:
# region -> step traces: one panel per swept parameter (population)
# --- step traces: one panel per swept parameter (each member = a jet line) ------------------
# Each panel: every member's parameter value vs backend iteration (jet), with the prior bounds
# (dotted grey). No target line -- parameters have no fixed target; the spread that remains at the
# final iteration is that parameter's identifiability.
paramTraces   = {p: chainSteps[:, :, j].T for j, p in enumerate(param_names)}   # {param:(nPop,nIter)}
rangesByParam = {p: (float(lo[j]), float(hi[j])) for j, p in enumerate(param_names)}

libPlots.plotCalibrationConvergence(
    paramTraces, traceT=None, ranges=rangesByParam, paramForObs=None, showLegend=False,
    ylim=runConfig["analysis"]["stepTraceYLim"], robustPct=tuple(runConfig["analysis"]["stepTracePct"]),
    divLimit=runConfig["analysis"]["divergenceLimit"],
    title=f"{method} step traces -- parameters (population, all iterations)")
plt.show()
# endregion

## Per-observation error boxplot

In [ ]:
# region -> boxplot: relative error distribution per observation (final population)
# --- per-observation relative error across the final-iteration population: the fit analogue of the
# batch/serial convergence-run boxplot, each population member playing the role of a "run". Dashed
# red lines mark the +/- errBand target-tolerance band. cmaes gens are NaN-padded to a common width,
# so each observation's column is filtered to its finite members.
errTarget = runConfig["analysis"].get("errBand", 2.0)
finalPop  = obsSteps[-1]                                         # (nPop, nTargeted) gauge obs, final iter
errRel    = (finalPop - tgt) / tgt * 100.0                      # (nPop, nTargeted) relative error %
cols      = [errRel[np.isfinite(errRel[:, k]), k] for k in range(errRel.shape[1])]
fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(cols, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=False)
ax.axhline(0.0, color="k", lw=0.8)
ax.axhline(errTarget, color="r", ls="--", lw=0.8, label=f"±{errTarget}% target")
ax.axhline(-errTarget, color="r", ls="--", lw=0.8)
ax.set_ylabel("relative error (%)")
ax.set_title(f"{method} convergence error across {finalPop.shape[0]} population members (final iteration)")
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.tight_layout()
plt.show()
# endregion

## Convergence / mixing diagnostics

<details>
<summary>Walker traces, integrated autocorrelation τ with N_eff, and split-R̂ across the walkers — evidence the chain mixed rather than stalling in a burn-in transient.</summary>

These are the same two numbers the Phase-1 early stop gates on (`emcee.rhatTol` / `emcee.neffMin`), so an early-stopped run satisfies them by construction and a capped run shows how far it got.

Reading the archived 256-walker file: max split-R̂ **2.58** with τ ≈ 130–240 over only 1900 steps (burn-in 400 ≈ 2τ) — the chain never mixed, which is why the MAP is good (2.1%) while the posterior *mean* is not (201%). That is an unconverged transient, not a multimodal posterior. The current config (64 walkers × 8000 steps, burn-in 2000 ≈ 10τ, warmup-cloud init, DE/snooker moves) targets ~40τ.

</details>

In [ ]:
# region -> ADD-3 mixing diagnostics: walker traces, autocorr/N_eff, split-Rhat
# Guards the posterior claim (R2-2 "convergence"): the chain must have MIXED, not stalled in a
# burn-in transient (the failure mode of the old hr_test run). Reads chain_steps (nIter,nWalkers,P).
# These are the SAME two numbers the Phase-1 early-stop gate watches (utils.splitRhat + N_eff), so a
# run that stopped early reports max split-Rhat < emcee.rhatTol here by construction; a run that hit
# the nSteps cap reports whatever it reached, and split-Rhat >> 1.05 means the posterior (not the MAP)
# is not yet trustworthy -- the archived 256-walker/1900-step file sits at 2.58.
burnIn = int(diag.get("burnIn", runConfig["inference"]["emcee"]["burnIn"]))
post   = chainSteps[burnIn:]                                    # (nPost, nWalkers, P)
nPost, nWalk, _ = post.shape
try:
    tau = np.asarray(emcee.autocorr.integrated_time(chainSteps, tol=0), float)
except Exception as e:
    tau = np.asarray(diag.get("tau", [np.nan] * len(param_names)), float); print("autocorr fallback:", e)
nEff = chain.shape[0] / tau
rhat = np.array([utils.splitRhat(post[:, :, j]) for j in range(len(param_names))])

mix = pd.DataFrame({"param": param_names, "tau": tau, "N_eff": nEff,
                    "split_Rhat": rhat}).set_index("param")
print(f"post-burn: {nPost} steps x {nWalk} walkers | accept {accept:.3f} | "
      f"min N_eff {np.nanmin(nEff):.0f} | max split-Rhat {np.nanmax(rhat):.3f}")

# --- stranded-walker audit -------------------------------------------------------------------
# A single walker parked far from the ensemble (emcee's DE proposals are too small to bring it back)
# holds split-Rhat off 1 on its own and skews every ensemble-mean readout, while the MAP and the
# posterior over the OTHER walkers are perfectly sound. Same criterion Phase 1 rescues on
# (emcee.rescue.logpDrop, in J = -2*logp units), so a run made with rescue enabled reports 0 here.
# The archived 8000-step file has exactly one: walker 36 at J ~ 22681 (4 moves in 6000 steps) --
# with it, max split-Rhat 441.7 and ensemble mean|rel| 2.42%; without it, 1.006 and 1.39%.
# Reported, NOT filtered: the corner plot and posterior above still show every walker.
Jw       = np.sum(((obsSteps[burnIn:] - tgt) / sigma) ** 2, axis=2)      # (nPost, nWalkers) chi-square
Jmed     = np.median(Jw)
stranded = np.where(np.median(Jw, axis=0) > Jmed + 2.0 * runConfig["inference"]["emcee"]["rescue"]["logpDrop"])[0]
print(f"stranded walkers: {stranded.size}/{nWalk} (median J {Jmed:.1f}; rescued in Phase 1: "
      f"{diag.get('nRescued', 'n/a')})")
if stranded.size:
    keep  = np.setdiff1d(np.arange(nWalk), stranded)
    rKept = np.nanmax([utils.splitRhat(post[:, keep, j]) for j in range(len(param_names))])
    print(f"  walkers {list(stranded)} at median J {[round(float(x)) for x in np.median(Jw, axis=0)[stranded]]} "
          f"-> max split-Rhat {np.nanmax(rhat):.1f} with them, {rKept:.3f} without")


# walker-trace panels for the loosest-mixing params (largest tau)
nTrace = min(runConfig["analysis"].get("nTraceParams", 6), len(param_names))
order  = np.argsort(-np.nan_to_num(tau, nan=-np.inf))[:nTrace]
fig, axes = plt.subplots(nTrace, 1, figsize=(9, 1.5 * nTrace), sharex=True)
axes = np.atleast_1d(axes)
for ax, j in zip(axes, order):
    ax.plot(chainSteps[:, :, j], color="C0", alpha=0.15, lw=0.6)
    ax.axvline(burnIn, color="C3", ls="--", lw=1.0)
    ax.set_ylabel(utils.labelFor(param_names[j], "latex"), rotation=0, ha="right", va="center")
axes[-1].set_xlabel("step (red dashed = burn-in cut)")
axes[0].set_title(f"{method} walker traces -- {nTrace} loosest-mixing params")
plt.tight_layout(); plt.show()
mix.round(3)
# endregion

In [ ]:
# region -> cost comparison: fit forward evals
print(f"{method}: {nEval} forward evals ({settleRuns} x {runTime} s each) -> {fitWall:.1f} s wall "
      f"| accept {accept:.3f} | diag {diag}")
# endregion

## Paper artifacts — appendix summary table + search-trace figure


<details>

The two artifacts for the manuscript's MCMC-baseline appendix subsection
(`app:mcmc`). The table mirrors the shape of the EFC convergence table
(`tab:summaryTable`) so the baseline reads against it row for row, with statistics
taken over the **full post-burn posterior** rather than one correlated snapshot of
the ensemble. The figure is the paper-sized counterpart of the full step-trace grid
above: only the four parameters that separate the gradient-descent baseline's two
basins, plus the residual history and the final-point scatter.

Both are written only when `runConfig["paper"]["emit"]` is `True`; flip it for one
run and revert, so a normal plot pass never rewrites the manuscript's files.

</details>


In [ ]:
# region -> paper artifacts: appendix summary table + compact search-trace figure
# Population for every statistic = the FULL post-burn chain (nSteps-burnIn steps x nWalkers), i.e.
# the posterior itself. obsTargeted[k] and param_names[k] are the controller PAIR (observable k is
# the target of parameter k), which is what lets the table carry both halves on one row exactly as
# tab:summaryTable does.
pc      = runConfig["paper"]
burnIn  = int(diag.get("burnIn", 0))
postObs = obsSteps[burnIn:].reshape(-1, obsSteps.shape[2])      # (S, nTargeted) gauge observations
errRel  = (postObs - tgt) / tgt * 100.0                         # signed relative error, %

fig = libPlots.plotSearchTraces(
    chainSteps, obsSteps, tgt, bounds, param_names,
    pc["traceParams"], scatterPair=pc["scatterPair"], groupMask=None,
    burnIn=burnIn, errBand=pc["errBand"], figSize=tuple(pc["figSize"]))
plt.show()

_meanAbs = np.abs(errRel.mean(axis=0)).max()
_cv      = 100.0 * chain.std(axis=0) / np.abs(chain.mean(axis=0))
_worstP  = param_names[int(np.argmax(_cv))]
_nSamp   = f"{chain.shape[0]:,}".replace(",", "\\,")            # LaTeX thousands separator
rows = {}
for k, (o, p) in enumerate(zip(obsTargeted, param_names)):
    rows[utils.labelFor(o, "latex")] = [
        f"{tgt[k]:.4g}", f"{errRel[:, k].mean():.3f}", f"{errRel[:, k].std():.3f}",
        utils.labelFor(p, "latex"), f"{chain[:, k].mean():.3f}", f"{chain[:, k].std():.3f}"]

latex = utils.generateLatexTableInline(
    rows, ["Vars.", "Targets", "Relative Error \\%", "std", "Param.", "Value", "std"],
    ref="tab:mcmcSummary",
    colSpec="C{0.7cm} C{0.7cm} |C{1.0cm} C{0.7cm}|C{0.7cm} C{0.7cm} C{0.7cm}",
    fontSize="small",
    caption=(
        "Ensemble-MCMC baseline calibration, reported in the same form as the Embedded "
        "Feedback Controller convergence test of Table~\\ref{tab:summaryTable}: the same sixteen "
        "model parameters driven to the same sixteen physiological targets, over the same prior "
        "box and under the same explicit-Euler integration. Statistics are taken over the full "
        f"post-burn posterior ({int(chainSteps.shape[0]) - burnIn} sampler steps $\\times$ "
        f"{int(chainSteps.shape[1])} walkers $=$ {_nSamp} samples). For each calibration "
        "target the table reports the prescribed target value, the mean signed relative error and "
        "its standard deviation over the posterior, together with the corresponding parameter's "
        f"posterior mean and standard deviation. The sampler holds every target to within "
        f"{_meanAbs:.2f}\\,\\% in the posterior mean and resolves every parameter to better than "
        f"{np.sort(_cv)[-2]:.1f}\\,\\% apart from {utils.labelFor(_worstP, 'latex')}, whose "
        f"posterior is {_cv.max():.1f}\\,\\% wide and is the one sloppy direction of the problem; "
        "the walker histories are shown in Figure~\\ref{fig:mcmcTraces}."))

print(f"max |mean rel err| over targets: {_meanAbs:.4f} % | widest posterior: "
      f"{_worstP} at CV {_cv.max():.2f} %")

if pc["emit"]:
    os.makedirs(pc["dir"], exist_ok=True)
    os.makedirs(pc["imageDir"], exist_ok=True)
    _t = os.path.join(pc["dir"], pc["table"])
    with open(_t, "w") as fh:
        fh.write(latex)
    print(f"wrote {_t}")
    _f = os.path.join(pc["imageDir"], pc["figure"])
    fig.savefig(_f, dpi=pc["dpi"], bbox_inches="tight")
    print(f"wrote {_f}")
else:
    print("paper.emit is False -- table/figure not written")
# endregion


In [ ]:
# region -> release GPU memory
import gc
for _v in ("chain", "chainSteps", "obsSteps", "lp", "prep", "equivObs"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion